# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pathakadithi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### My baseline rule

I will rank pages for a **refresh action** using historical staleness and CTR-vs-position signals. Pages that appear stale and have weaker CTR than expected for their historical search position receive a higher refresh score. Pages with stronger signals receive a lower score.

The rule uses only historical information available up to the scoring point and does not use future performance or label-derived information.

### Reason code

* `refresh_opportunity` — the page has historical signals suggesting that its content may benefit from a refresh.


In [ ]:
# Signal Check 1: CTR vs historical search position

signal1 = df.copy()

# Keep rows with a valid historical position and impressions
signal1 = signal1[
    signal1["avg_position_90d"].notna()
    & (signal1["impressions_90d"] > 0)
].copy()

# Position buckets
signal1["position_bucket"] = pd.cut(
    signal1["avg_position_90d"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Bucket summary
signal1_table = (
    signal1.groupby("position_bucket", observed=False)
    .agg(
        n=("ctr_90d", "size"),
        mean_ctr=("ctr_90d", "mean"),
        median_ctr=("ctr_90d", "median")
    )
    .reset_index()
)

print(signal1_table)

  position_bucket       n  mean_ctr  median_ctr
0             1-3  405442  0.004195         0.0
1            4-10  978489  0.002434         0.0
2           11-20  327380  0.001565         0.0
3             21+  702937  0.000438         0.0


**Signal 1 verdict: CONFIRMED**

CTR decreases consistently as historical search position gets worse: mean CTR falls from 0.004195 for positions 1–3 to 0.000438 for positions 21+. This supports using CTR-vs-position as a baseline signal. The large sample sizes in every bucket make the pattern clear, although the median CTR is 0 in every bucket, so the mean is carrying the observed difference.


In [ ]:
# Signal Check 2: Recent performance decline

signal2 = df.copy()

# Keep rows where both recent and previous 30-day impressions are available
signal2 = signal2[
    (signal2["impressions_last30"] > 0)
    & (signal2["impressions_prev30"] > 0)
].copy()

# Calculate CTR for each 30-day period
signal2["ctr_last30"] = (
    signal2["clicks_last30"] / signal2["impressions_last30"]
)

signal2["ctr_prev30"] = (
    signal2["clicks_prev30"] / signal2["impressions_prev30"]
)

# Change in CTR: negative means CTR declined recently
signal2["ctr_change"] = (
    signal2["ctr_last30"] - signal2["ctr_prev30"]
)

# Create decline buckets
signal2["ctr_change_bucket"] = pd.cut(
    signal2["ctr_change"],
    bins=[-np.inf, -0.002, -0.0005, 0, np.inf],
    labels=["large decline", "small decline", "no decline", "improved"],
    include_lowest=True
)

signal2_table = (
    signal2.groupby("ctr_change_bucket", observed=False)
    .agg(
        n=("ctr_change", "size"),
        mean_ctr_change=("ctr_change", "mean"),
        median_ctr_change=("ctr_change", "median")
    )
    .reset_index()
)

print(signal2_table)

  ctr_change_bucket        n  mean_ctr_change  median_ctr_change
0     large decline    72668    -5.932688e-02          -0.023441
1     small decline     4158    -1.263391e-03          -0.001270
2        no decline  1564300    -1.772962e-07           0.000000
3          improved    63641     6.243288e-02           0.022222


**Signal 2 verdict: CONFIRMED**

Recent CTR change provides a useful deterioration signal. The large-decline bucket has a mean CTR change of -0.0593, while the improved bucket has a mean change of +0.0624. This supports using recent CTR decline as part of the refresh baseline. However, most rows are in the no-decline bucket, so this signal should not be treated as sufficient by itself.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

df = dataset["train"].to_pandas()

df["ctr_90d"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    0
)

df["ctr_last30"] = np.where(
    df["impressions_last30"] > 0,
    df["clicks_last30"] / df["impressions_last30"],
    np.nan
)

df["ctr_prev30"] = np.where(
    df["impressions_prev30"] > 0,
    df["clicks_prev30"] / df["impressions_prev30"],
    np.nan
)

print("Rows:", len(df))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Rows: 2414248


In [ ]:
import pandas as pd
import numpy as np

df = dataset["train"].to_pandas()

# Historical CTR
df["ctr_90d"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    0
)

# Recent 30-day CTR
df["ctr_last30"] = np.where(
    df["impressions_last30"] > 0,
    df["clicks_last30"] / df["impressions_last30"],
    0
)

# Previous 30-day CTR
df["ctr_prev30"] = np.where(
    df["impressions_prev30"] > 0,
    df["clicks_prev30"] / df["impressions_prev30"],
    0
)

# Historical change in search position
# Positive = position became worse
df["position_change"] = (
    df["avg_position_last30"] - df["avg_position_prev30"]
)

print("Rows:", len(df))

print(df[[
    "impressions_90d",
    "clicks_90d",
    "ctr_90d",
    "ctr_last30",
    "ctr_prev30",
    "avg_position_90d",
    "avg_position_last30",
    "avg_position_prev30",
    "position_change"
]].head())

Rows: 2414248
   impressions_90d  clicks_90d  ctr_90d  ctr_last30  ctr_prev30  \
0               11           0      0.0         0.0         0.0   
1               13           0      0.0         0.0         0.0   
2               16           0      0.0         0.0         0.0   
3               55           0      0.0         0.0         0.0   
4               14           0      0.0         0.0         0.0   

   avg_position_90d  avg_position_last30  avg_position_prev30  position_change  
0         10.818182                  NaN            10.818182              NaN  
1          1.769231                  NaN            11.000000              NaN  
2         23.562500            24.272727            22.000000         2.272727  
3          2.200000            13.000000             0.000000        13.000000  
4          3.428571                  NaN                  NaN              NaN  


In [ ]:
# ============================================================
# Baseline ranked queue
# ============================================================

queue = df.copy()

# ------------------------------------------------------------
# 1. CTR-vs-position component
# ------------------------------------------------------------

queue["position_bucket"] = pd.cut(
    queue["avg_position_90d"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

# Historical average CTR for each position bucket
position_ctr = (
    queue.groupby("position_bucket", observed=False)["ctr_90d"]
    .mean()
)

# Convert expected CTR explicitly to numeric
queue["expected_ctr"] = (
    queue["position_bucket"]
    .map(position_ctr)
    .astype(float)
)

# CTR below expected position-based CTR
queue["ctr_gap"] = (
    queue["expected_ctr"] - queue["ctr_90d"]
).fillna(0)

# ------------------------------------------------------------
# 2. Recent CTR decline
# ------------------------------------------------------------

queue["recent_ctr_decline"] = (
    queue["ctr_prev30"] - queue["ctr_last30"]
).fillna(0)

# ------------------------------------------------------------
# 3. Normalize components
# ------------------------------------------------------------

ctr_gap_scale = queue["ctr_gap"].quantile(0.95)
decline_scale = queue["recent_ctr_decline"].quantile(0.95)

if ctr_gap_scale > 0:
    queue["ctr_gap_score"] = (
        queue["ctr_gap"] / ctr_gap_scale
    ).clip(0, 1)
else:
    queue["ctr_gap_score"] = 0.0

if decline_scale > 0:
    queue["decline_score"] = (
        queue["recent_ctr_decline"] / decline_scale
    ).clip(0, 1)
else:
    queue["decline_score"] = 0.0

# ------------------------------------------------------------
# 4. ONE baseline score
# ------------------------------------------------------------

queue["baseline_score"] = (
    0.5 * queue["ctr_gap_score"]
    + 0.5 * queue["decline_score"]
)

# ------------------------------------------------------------
# 5. ONE reason code + action
# ------------------------------------------------------------

queue["reason_code"] = "refresh_opportunity"

queue["action"] = np.where(
    queue["baseline_score"] > 0,
    "refresh",
    "no_action"
)

# ------------------------------------------------------------
# 6. Rank everything
# ------------------------------------------------------------

queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ------------------------------------------------------------
# 7. Output table
# ------------------------------------------------------------

baseline_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "query_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

print("Queue rows:", len(baseline_queue))
print("\nTop 10:")
print(baseline_queue.head(10).to_string(index=False))

Queue rows: 2414248

Top 10:
 rank          client_hash_id          content_hash_id          query_hash_id  baseline_score         reason_code  action
    1 client_e547b89c05043229 content_17608f489483f8f8 query_fc3c1abcfdeb2695             0.5 refresh_opportunity refresh
    2 client_e547b89c05043229 content_17608f489483f8f8 query_f08306770da625b9             0.5 refresh_opportunity refresh
    3 client_08a6a72ff48e62c0 content_447894f2faf0d2bc query_922b8eca2a24cd34             0.5 refresh_opportunity refresh
    4 client_e547b89c05043229 content_17608f489483f8f8 query_d78ce04b25889571             0.5 refresh_opportunity refresh
    5 client_e547b89c05043229 content_17608f489483f8f8 query_d238f425d28c2a57             0.5 refresh_opportunity refresh
    6 client_e547b89c05043229 content_17608f489483f8f8 query_cf806511a630a219             0.5 refresh_opportunity refresh
    7 client_e547b89c05043229 content_17608f489483f8f8 query_bcd31c3902bf89c3             0.5 refresh_opportunity ref

In [ ]:
# Write the ranked baseline queue to the required output path

from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"CSV written to: {output_path}")
print(f"Rows written: {len(baseline_queue)}")

CSV written to: work/outputs/baseline_action_score.csv
Rows written: 2414248


In [ ]:
check = pd.read_csv("work/outputs/baseline_action_score.csv")

print(check.shape)
print(check.head(10))
print("\nActions:")
print(check["action"].value_counts())

(2414248, 7)
   rank           client_hash_id           content_hash_id  \
0     1  client_e547b89c05043229  content_17608f489483f8f8   
1     2  client_e547b89c05043229  content_17608f489483f8f8   
2     3  client_08a6a72ff48e62c0  content_447894f2faf0d2bc   
3     4  client_e547b89c05043229  content_17608f489483f8f8   
4     5  client_e547b89c05043229  content_17608f489483f8f8   
5     6  client_e547b89c05043229  content_17608f489483f8f8   
6     7  client_e547b89c05043229  content_17608f489483f8f8   
7     8  client_08a6a72ff48e62c0  content_4483e354a2992497   
8     9  client_e547b89c05043229  content_17608f489483f8f8   
9    10  client_e547b89c05043229  content_17608f489483f8f8   

            query_hash_id  baseline_score          reason_code   action  
0  query_fc3c1abcfdeb2695             0.5  refresh_opportunity  refresh  
1  query_f08306770da625b9             0.5  refresh_opportunity  refresh  
2  query_922b8eca2a24cd34             0.5  refresh_opportunity  refresh  
3  query

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Top-20 review data

top20 = queue.head(20)[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "baseline_score",
    "ctr_90d",
    "expected_ctr",
    "ctr_gap",
    "ctr_gap_score",
    "ctr_last30",
    "ctr_prev30",
    "recent_ctr_decline",
    "decline_score",
    "reason_code",
    "action"
]].copy()

print(top20.to_string(index=False))

 rank          client_hash_id          content_hash_id          query_hash_id  baseline_score  ctr_90d  expected_ctr  ctr_gap  ctr_gap_score  ctr_last30  ctr_prev30  recent_ctr_decline  decline_score         reason_code  action
    1 client_e547b89c05043229 content_17608f489483f8f8 query_fc3c1abcfdeb2695             0.5      0.0      0.004195 0.004195            1.0         0.0         0.0                 0.0            0.0 refresh_opportunity refresh
    2 client_e547b89c05043229 content_17608f489483f8f8 query_f08306770da625b9             0.5      0.0      0.004195 0.004195            1.0         0.0         0.0                 0.0            0.0 refresh_opportunity refresh
    3 client_08a6a72ff48e62c0 content_447894f2faf0d2bc query_922b8eca2a24cd34             0.5      0.0      0.004195 0.004195            1.0         0.0         0.0                 0.0            0.0 refresh_opportunity refresh
    4 client_e547b89c05043229 content_17608f489483f8f8 query_d78ce04b25889571           

### Top-20 review

1. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 while the historical position-bucket expectation is 0.004195. | **What would make it wrong:** The query may have very low exposure or the zero CTR may be caused by missing recent traffic rather than stale content.

2. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 versus the position-based expectation of 0.004195. | **What would make it wrong:** The query may have insufficient impressions or the page may not actually need a content refresh.

3. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 and below the historical position-based expectation. | **What would make it wrong:** The recent-period data is incomplete, so the baseline cannot confirm that the problem is current.

4. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 versus an expected CTR of 0.004195. | **What would make it wrong:** A zero CTR may reflect limited opportunity rather than content quality.

5. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The historical CTR gap is positive and receives the maximum normalized gap score. | **What would make it wrong:** The query may have too little traffic to justify a refresh.

6. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 and the position-based expected CTR is 0.004195. | **What would make it wrong:** Recent CTR is unavailable, so there is no evidence of recent deterioration.

7. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The row has zero historical CTR and a positive CTR gap. | **What would make it wrong:** Missing recent observations could make the refresh recommendation unreliable.

8. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 compared with the position-bucket expectation of 0.004195. | **What would make it wrong:** The zero CTR could result from low exposure rather than a refresh opportunity.

9. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 and recent CTR is also 0, producing a strong historical CTR-gap signal. | **What would make it wrong:** The row has no evidence of improvement or deterioration, so the rule may be reacting only to zero CTR.

10. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The historical CTR is below the expected CTR for the position bucket. | **What would make it wrong:** Zero recent CTR does not by itself prove that content needs updating.

11. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The row has zero CTR and a maximum CTR-gap component. | **What would make it wrong:** Recent performance is unavailable, so the recommendation could be driven by sparse observations.

12. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 against an expected CTR of 0.004195. | **What would make it wrong:** The query may not have enough recent impressions to support a refresh decision.

13. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The historical CTR gap is positive and drives the score. | **What would make it wrong:** The low CTR may reflect limited demand rather than stale content.

14. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — Zero historical CTR creates a large gap from the position-based expectation. | **What would make it wrong:** Missing recent observations prevent confirmation that the issue is ongoing.

15. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The page/query has zero CTR despite a positive historical position-based expectation. | **What would make it wrong:** Low traffic or query-specific behavior could explain the zero CTR without requiring a refresh.

16. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The score is driven by the maximum normalized CTR-gap component. | **What would make it wrong:** Recent performance is unavailable, making the recommendation less certain.

17. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — Historical CTR is 0 versus the position-based expectation of 0.004195. | **What would make it wrong:** The observed zero CTR may come from sparse query exposure rather than stale content.

18. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — Both historical and recent CTR are 0, while the position bucket has a positive expected CTR. | **What would make it wrong:** The signal does not show recent decline, so the refresh action may be too aggressive.

19. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — The row has zero historical CTR and a large position-based CTR gap. | **What would make it wrong:** The previous-period data is incomplete, so there is insufficient evidence of an active content problem.

20. **Action:** refresh | **Reason code:** `refresh_opportunity` | **Confidence:** Moderate — CTR is 0 compared with the position-based expected CTR of 0.004195. | **What would make it wrong:** Low query volume or missing recent observations could make the refresh recommendation incorrect.

### Review takeaway

The top 20 are highly similar: all receive a score of 0.5 because the CTR-gap component is maximal while the recent-decline component contributes 0. This is a useful weakness of the baseline to document: **the rule can prioritize zero-CTR rows even when recent performance is unavailable, so the refresh action should be treated as a baseline hypothesis rather than proof that content is stale.**


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Weak picks

The weakest-looking picks are the top rows that have **0 CTR but missing recent-period CTR values**. They receive a high score because their historical CTR is below the position-bucket expectation, but the baseline has no recent evidence that the performance is currently declining.

The top 20 are also heavily tied at a score of **0.5**, so the exact ordering among these rows should not be interpreted as meaningful. This is a limitation of the simple baseline and is a useful weakness for the Week-5 model to improve.

A particularly weak case is a row with zero historical CTR and no recent observations: the rule treats the zero CTR as an opportunity, but the data cannot distinguish a genuine content problem from insufficient recent exposure.

### Leakage check

* I did **not** use FlyRank product flags such as `health_score`, `needs_ctr_fix`, `is_quick_win`, or `recommended_action` as features.
* I did **not** use a future performance window or future outcome.
* The score uses historical `ctr_90d`, historical search-position buckets, and the available recent/previous 30-day CTR observations.
* The reason code `refresh_opportunity` is my own baseline label; it is not copied from a FlyRank product flag.
* Hashed client/content/query IDs are used only for identifying and reviewing rows, not as scoring features.
* The ranked queue is therefore intended as a **historical, decision-support baseline**, not proof that a page needs a refresh.

FlyRank's published guidance specifically warns against using product flags as model features because they can leak the decision signal.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

* [x] Every section above is filled with markdown reasoning and supporting code.
* [ ] The notebook runs top to bottom with no errors — verified by running **Runtime → Run all**.
* [x] No client names, URLs, or private queries are included; only anonymized hashed IDs are used for row review.
* [x] Claims use careful language such as observed, measured, directional, and decision-support.
* [ ] The notebook has been committed to my repository under `work/notebooks/`; I will verify the commit and submit the repository URL.

### Final baseline notes

The baseline is intentionally simple and transparent. The observed signal checks support CTR-vs-position and recent CTR decline as useful historical signals. The top-ranked rows also expose weaknesses: many are tied and several have missing recent observations. These limitations are documented rather than hidden and provide a clear baseline for the Week-5 model to improve.
